In [1]:
from datasets import load_dataset
from langchain.schema import Document

# Load dataset using the parquet revision
try:
    dataset = load_dataset("neulab/ted_multi", split="train[:1000]", revision="refs/convert/parquet")
except Exception as e:
    print(f"Revision approach failed: {e}")
    try:
        dataset = load_dataset("neulab/ted_multi", data_files="*.parquet", split="train[:1000]")
    except Exception as e2:
        print(f"Parquet pattern approach failed: {e2}")
        dataset = load_dataset("neulab/ted_multi", split="train[:1000]", trust_remote_code=True)

documents = []
for item in dataset:
    # Extract language and translation lists
    langs = item["translations"]["language"]
    texts = item["translations"]["translation"]
    
    # Find indices where language is nl and en
    try:
        idx_nl = langs.index("nl")
        idx_en = langs.index("en")
    except ValueError:
        # Skip if either language is not available
        continue
    
    nl_text = texts[idx_nl]
    en_text = texts[idx_en]
    
    if en_text:
        doc = Document(
            page_content=en_text,
            metadata={
                "source_text": nl_text,
                "translation_pair": "nl_en",
                "talk_name": item["talk_name"]
            }
        )
        documents.append(doc)
#print (documents)
print(f"Created {len(documents)} documents from TED talks with EN-NL translation pairs")

Created 644 documents from TED talks with EN-NL translation pairs


In [2]:
# Step 1: Split documents into smaller chunks for better retrieval
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialize text splitter with reasonable chunk size
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Characters per chunk
    chunk_overlap=50,  # Overlap to maintain context
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

# Split the documents
chunks = text_splitter.split_documents(documents)
print(f"Split {len(documents)} documents into {len(chunks)} chunks")
print(f"\nExample chunk:")
print(f"Content: {chunks[0].page_content[:200]}...")
print(f"Metadata: {chunks[0].metadata}")

Split 644 documents into 644 chunks

Example chunk:
Content: Amongst all the troubling deficits we struggle with today — we think of financial and economic primarily — the ones that concern me most is the deficit of political dialogue — our ability to address m...
Metadata: {'source_text': 'Van al onze verontrustende tekorten tegenwoordig — als eerste denk je aan financiële en economische — is het meest belangrijke voor mij het tekort aan politieke dialoog : ons onvermogen met moderne conflicten om te gaan , naar de kern te gaan en de sleutelfiguren te begrijpen , en ze aan te pakken .', 'translation_pair': 'nl_en', 'talk_name': 'jonas_gahr_store_in_defense_of_dialogue'}


In [3]:
# Step 2: Generate embeddings and create FAISS vector database
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize HuggingFace embeddings model
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",  # Fast and efficient model
    model_kwargs={'device': 'cpu'},  # Use 'cuda' if GPU available
    encode_kwargs={'normalize_embeddings': True}  # Normalize for cosine similarity
)

# Create FAISS vector store from document chunks
print("Creating FAISS vector store...")
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"✓ Vector store created with {len(chunks)} embedded chunks")
print(f"✓ Embedding dimension: {len(embeddings.embed_query('test'))}")

Loading embedding model...


/tmp/ipykernel_1576327/65899638.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
2025-10-14 19:44:53.600749: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-14 19:44:53.718633: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VN

Creating FAISS vector store...
✓ Vector store created with 644 embedded chunks
✓ Embedding dimension: 384


In [4]:
# Check Python environment and install missing packages
import sys
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")

# Install required packages in the current notebook kernel
import subprocess

packages = [
    "sentence-transformers",
    "faiss-cpu",
    "transformers"
]

for package in packages:
    try:
        __import__(package.replace("-", "_"))
        print(f"✓ {package} is already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} installed successfully")

Python executable: /usr/bin/python3
Python version: 3.13.7 (main, Aug 14 2025, 00:00:00) [GCC 14.3.1 20250523 (Red Hat 14.3.1-1)]
✓ sentence-transformers is already installed
Installing faiss-cpu...
✓ faiss-cpu installed successfully
✓ transformers is already installed


In [5]:
# Step 3: Configure retriever for semantic search
retriever = vectorstore.as_retriever(
    search_type="similarity",  # Use similarity search
    search_kwargs={"k": 4}  # Retrieve top 4 most relevant chunks
)

# Test the retriever with a sample query
test_query = "What are the impacts of climate change?"
retrieved_docs = retriever.get_relevant_documents(test_query)

print(f"✓ Retriever configured successfully")
print(f"\nTest query: '{test_query}'")
print(f"Retrieved {len(retrieved_docs)} relevant documents:\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"Document {i}:")
    print(f"  Content: {doc.page_content[:150]}...")
    print(f"  Talk: {doc.metadata['talk_name']}\n")

✓ Retriever configured successfully

Test query: 'What are the impacts of climate change?'
Retrieved 4 relevant documents:

Document 1:
  Content: How are we going to solve climate change through negotiations , unless we are able to make civil society and people , not part of the problem , but pa...
  Talk: jonas_gahr_store_in_defense_of_dialogue

Document 2:
  Content: And we have to go also beyond traditional diplomacy to the survival issue of our times , climate change ....
  Talk: jonas_gahr_store_in_defense_of_dialogue

Document 3:
  Content: It is going to demand an inclusive process of diplomacy very different from the one we are practicing today as we are heading to new rounds of difficu...
  Talk: jonas_gahr_store_in_defense_of_dialogue

Document 4:
  Content: That is having real impact in a country like Ethiopia , and it &apos;s why you see their child mortality numbers coming down 25 percent from 2000 to 2...
  Talk: melinda_french_gates_what_nonprofits_can_learn_from_coca_c

/tmp/ipykernel_1576327/4154690822.py:9: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retrieved_docs = retriever.get_relevant_documents(test_query)


In [6]:
# Install OpenAI package if needed (for LM Studio compatibility)
import sys
import subprocess

try:
    import openai
    from langchain_openai import ChatOpenAI
    print("✓ OpenAI packages already installed")
except ImportError:
    print("Installing openai and langchain-openai packages...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openai", "langchain-openai", "-q"])
    print("✓ Packages installed successfully")

# Set dummy API key for LM Studio (doesn't require real authentication)
import os
os.environ["OPENAI_API_KEY"] = "lm-studio"
print("✓ Dummy API key set for LM Studio compatibility")

✓ OpenAI packages already installed
✓ Dummy API key set for LM Studio compatibility


In [7]:
# Step 4: Connect to LM Studio and create RetrievalQA pipeline
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
import os

# Configure connection to LM Studio (OpenAI-compatible API)
print("Connecting to LM Studio...")

# Connect to LM Studio running on 127.0.0.1:1234
llm = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1",  # LM Studio API endpoint
    api_key="lm-studio",  # LM Studio doesn't require a real key
    model="bytedance/seed-oss-36b",  # Use the actual model name from LM Studio
    temperature=0.7,
    max_tokens=512,
    streaming=False
)

print("✓ Connected to LM Studio at http://127.0.0.1:1234")
print("  Using model: bytedance/seed-oss-36b")

# Create a prompt template optimized for instruction-tuned models
prompt_template = """You are a helpful assistant answering questions based on TED Talk transcripts.

Use the following context to answer the question. Provide a detailed and informative answer based on the context provided. If the context doesn't fully answer the question, say so.

Context:
{context}

Question: {question}

Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# Create the RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

print("✓ RetrievalQA pipeline created with LM Studio model")

Connecting to LM Studio...
✓ Connected to LM Studio at http://127.0.0.1:1234
  Using model: bytedance/seed-oss-36b
✓ RetrievalQA pipeline created with LM Studio model


In [8]:
# Step 5: Test the RAG system with open-domain questions

# Define test questions on different topics
test_questions = [
    "What are the main challenges of climate change?",
    "How can education transform society?",
    "What role does technology play in addressing environmental issues?",
    "How can we improve access to education globally?"
]

print("=" * 80)
print("TESTING RAG QUESTION ANSWERING SYSTEM")
print("=" * 80)

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*80}")
    print(f"QUESTION {i}: {question}")
    print(f"{'='*80}")
    
    # Get answer from the QA chain
    result = qa_chain({"query": question})
    
    # Display the answer
    print(f"\nANSWER:")
    print(result['result'])
    
    # Display source documents
    print(f"\nSOURCE DOCUMENTS ({len(result['source_documents'])} retrieved):")
    for j, doc in enumerate(result['source_documents'], 1):
        print(f"\n  Source {j}:")
        print(f"  Talk: {doc.metadata['talk_name']}")
        print(f"  Content: {doc.page_content[:200]}...")
    
    print("\n" + "-"*80)

TESTING RAG QUESTION ANSWERING SYSTEM

QUESTION 1: What are the main challenges of climate change?


/tmp/ipykernel_1576327/3646433915.py:21: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain({"query": question})



ANSWER:
<seed:think>
Got it, let's tackle this question. The user is asking about the main challenges of climate change based on the provided context. First, I need to carefully look through the context to see what's mentioned.

The context starts by talking about solving climate change through negotiations but emphasizes that civil society and people need to be part of the solution, not just part of the problem. Then it mentions going beyond traditional diplomacy to the "survival issue" of climate change, demanding an inclusive process different from current practice, especially heading into new rounds of difficult negotiations. There's also a brief example of sanitation, but that's just an example, not a challenge.

Wait, the question is about the main challenges. The context doesn't explicitly list "main challenges" like rising temperatures, sea levels, etc.—those are common challenges but not in the text here. What the context does focus on is the approach to addressing climate ch

In [9]:
# Helper function to ask custom questions
def ask_question(question):
    """
    Ask a question to the RAG system and display the answer with sources.
    
    Args:
        question (str): The question to ask
    """
    print(f"\nQUESTION: {question}")
    print("="*80)
    
    result = qa_chain({"query": question})
    
    print(f"\nANSWER:")
    print(result['result'])
    
    print(f"\nSOURCES:")
    for i, doc in enumerate(result['source_documents'], 1):
        print(f"\n  [{i}] {doc.metadata['talk_name']}")
        print(f"      {doc.page_content[:150]}...")
    
    return result

# Example usage:
# ask_question("What innovations can help solve climate problems?")

# RAG System Summary

## System Architecture

This Retrieval-Augmented Generation (RAG) system combines semantic search with a transformer model to answer questions based on TED Talk transcripts.

### Components:

1. **Data Source**: TED Multi dataset (neulab/ted_multi)
   - Loaded 644 English-Dutch translation pairs from 1000 talks
   - Each document contains talk transcript and metadata

2. **Text Processing**:
   - RecursiveCharacterTextSplitter
   - Chunk size: 500 characters
   - Overlap: 50 characters

3. **Embeddings**: HuggingFace sentence-transformers/all-MiniLM-L6-v2
   - Fast and efficient embedding model
   - 384-dimensional embeddings
   - Normalized for cosine similarity

4. **Vector Store**: FAISS (Facebook AI Similarity Search)
   - Efficient similarity search
   - Stores all document chunks with embeddings

5. **Retriever**: Similarity-based retrieval
   - Returns top 4 most relevant chunks per query
   - Uses cosine similarity for ranking

6. **Language Model**: google/flan-t5-small
   - Instruction-tuned T5 model
   - Generates answers based on retrieved context

7. **QA Chain**: RetrievalQA with "stuff" chain type
   - Combines retrieved documents into single prompt
   - Returns both answer and source documents

### Usage:

```python
# Ask a question
result = ask_question("Your question here")

# Or use the qa_chain directly
result = qa_chain({"query": "Your question"})
print(result['result'])  # The answer
print(result['source_documents'])  # The sources
```